# Muon toy benchmark: larger weight-space divergence with lower function-space divergence proxies

This notebook is the presentation and analysis companion to `run_experiment.py` for the pair
`MUON_PARADOX_chaos_weights_order_functions`.

## Truthful scope

- This is a **toy synthetic benchmark**, not a production-scale optimizer evaluation.
- The script compares Muon and momentum SGD on small 4-layer linear and ReLU networks.
- "Function-space" quantities here are **proxy distances on a fixed test batch** `X_test`.
- The current implementation can support claims like:
  - Muon can show **larger weight-space divergence** than SGD in this setup, while
  - mapping **less of that divergence into the current function-space proxy metrics**.
- The current implementation does **not** prove global contraction, resolve a paradox, or establish a gauge-fixing mechanism.
    


## Notebook plan

1. Load the experiment module in a notebook-safe way.
2. Record reproducibility, runtime, and configuration information.
3. Run the script's structured `run_experiment(...)` entrypoint rather than duplicating the experiment core.
4. Show the saved figures inline.
5. Summarize the main proxy metrics, loss trajectories, and heuristic checks.
6. End with a calibrated conclusion that states what this toy benchmark supports and what it does not.
    


In [ ]:
import os
import time
import platform
import importlib.util
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import Image, Markdown, display
except ImportError:
    class Markdown(str):
        pass

    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
            self.kwargs = kwargs

        def __repr__(self):
            return f'Image(filename={self.filename!r})'

    def display(obj):
        print(obj)

try:
    import pandas as pd
except ImportError:
    pd = None

EXPERIMENT_RELATIVE = Path('experiments/Tier_1_Core_Mechanism_Tests/MUON_PARADOX_chaos_weights_order_functions')
EXPERIMENT_NAME = 'MUON_PARADOX_chaos_weights_order_functions'


def find_experiment_dir(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        direct = candidate / 'run_experiment.py'
        nested = candidate / EXPERIMENT_RELATIVE / 'run_experiment.py'
        if direct.exists() and candidate.name == EXPERIMENT_NAME:
            return candidate
        if nested.exists():
            return nested.parent
    raise FileNotFoundError(
        'Could not locate the experiment directory from the current working directory. '
        'Please start Jupyter inside the repository or cd into the repository before running this notebook.'
    )


EXPERIMENT_DIR = find_experiment_dir()
REPO_ROOT = EXPERIMENT_DIR.parents[2]
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'


def load_module(script_path):
    spec = importlib.util.spec_from_file_location('muon_paradox_experiment', script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


muon_exp = load_module(SCRIPT_PATH)


def as_frame(rows):
    if pd is None:
        return rows
    return pd.DataFrame(rows)


def show_rows(rows, title=None):
    if title:
        display(Markdown(f'#### {title}'))
    display(as_frame(rows))


print(f'Repo root: {REPO_ROOT}')
print(f'Experiment dir: {EXPERIMENT_DIR}')
print(f'Loaded script: {SCRIPT_PATH.name}')
    


In [ ]:
RUN_MODE = os.environ.get('MUON_NOTEBOOK_RUN_MODE', 'full').strip().lower()
if RUN_MODE not in {'full', 'smoke'}:
    raise ValueError(f'Unsupported RUN_MODE={RUN_MODE!r}; expected "full" or "smoke".')

VERBOSE = os.environ.get('MUON_NOTEBOOK_VERBOSE', '1') != '0'
MAKE_PLOTS = True
CONFIG_OVERRIDES = dict(muon_exp.SMOKE_TEST_CONFIG) if RUN_MODE == 'smoke' else None

print(f'Notebook run mode: {RUN_MODE}')
print(f'Progress logging enabled: {VERBOSE}')
if CONFIG_OVERRIDES is None:
    print('Using the script defaults.')
else:
    print('Using smoke-test overrides:')
    print(CONFIG_OVERRIDES)
    


In [ ]:
notebook_start_time = time.time()
results = muon_exp.run_experiment(
    config_overrides=CONFIG_OVERRIDES,
    output_dir=EXPERIMENT_DIR,
    make_plots=MAKE_PLOTS,
    verbose=VERBOSE,
)
notebook_wall_clock_seconds = time.time() - notebook_start_time

print(f'Notebook wall-clock runtime: {notebook_wall_clock_seconds:.2f} s')
print(f'Script-reported runtime: {results["runtime_seconds"]:.2f} s')
print('Top-level result keys:', sorted(results.keys()))
    


## 1. Counterpart identity, reproducibility, and runtime

The notebook now uses the script's structured return object instead of duplicating the experiment logic.
This keeps the pair aligned: `run_experiment.py` computes the benchmark; the notebook explains and presents it.
    


In [ ]:
identity_rows = [{
    'benchmark': results['identity']['benchmark'],
    'scope': results['identity']['scope'],
    'claim_scope': results['identity']['claim_scope'],
    'script_path': results['identity']['script_path'],
    'run_mode': RUN_MODE,
    'cwd': str(Path.cwd().resolve()),
    'repo_root': str(REPO_ROOT),
    'experiment_dir': str(EXPERIMENT_DIR),
    'python_version': platform.python_version(),
    'numpy_version': np.__version__,
    'notebook_wall_clock_seconds': round(notebook_wall_clock_seconds, 2),
    'experiment_runtime_seconds_reported': round(results['runtime_seconds'], 2),
    'plot_output_dir': results['artifacts']['output_dir'],
}]
show_rows(identity_rows, title='Run identity and environment')

config_rows = [{'parameter': key, 'value': value} for key, value in results['config'].items()]
seed_rows = [{'seed_name': key, 'value': value} for key, value in results['seeds'].items()]
show_rows(config_rows, title='Configuration used')
show_rows(seed_rows, title='Seeds used')
    


## 2. SGD learning-rate selection

The script preserves the original one-sided SGD tuning rule: it searches a fixed candidate list and picks the first learning rate that stays numerically stable under a coarse criterion.

This is a **stability heuristic**, not a symmetric hyperparameter search and not a matched-performance tuning procedure.
    


In [ ]:
lr_overview = results['summary_tables']['lr_selection']
show_rows(lr_overview, title='Chosen learning rates')

for net_type in muon_exp.NETWORK_TYPES:
    trial_rows = []
    for trial in results['lr_selection'][net_type]['trials']:
        trial_rows.append({
            'lr': trial['lr'],
            'stable': trial['stable'],
            'initial_loss': trial['initial_loss'],
            'final_loss': trial['final_loss'],
            'steps_completed': trial['steps_completed'],
        })
    show_rows(trial_rows, title=f'{net_type.upper()} SGD LR trials')
    


## 3. Main saved figures

The script still produces the benchmark figures as PNG artifacts. The notebook displays those saved outputs inline so the analysis remains easy to read in one place.
    


In [ ]:
plot_paths = results['artifacts']['plot_paths']
for key in ['combined', 'linear', 'relu']:
    path = plot_paths.get(key)
    if path and Path(path).exists():
        display(Markdown(f'### Figure: {key}'))
        display(Image(filename=path))
    else:
        print(f'Plot missing or not generated: {key}')
    


## 4. Face 1: perturbation-based divergence proxies

Face 1 starts from one shared initialization, adds small perturbations, and compares the resulting trajectories under SGD and Muon.

Important scope note:
- The finite-time Lyapunov-like quantities here are **trajectory-endpoint proxy summaries**, not theorem-level Lyapunov exponents.
- The function-space metric is evaluated on a fixed `X_test`, so it is a **toy output-space proxy**, not an exact global function metric.
    


In [ ]:
show_rows(results['summary_tables']['face1'], title='Face 1 summary metrics')
    


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, net_type in zip(axes, muon_exp.NETWORK_TYPES):
    face1 = results['networks'][net_type]['face1']
    ax.plot(face1['base']['sgd']['loss_trajectory'], label='SGD', color='#4477AA', linewidth=2)
    ax.plot(face1['base']['muon']['loss_trajectory'], label='Muon', color='#CC3311', linewidth=2)
    ax.set_title(f'{net_type.upper()} net: base loss trajectories')
    ax.set_xlabel('Training step')
    ax.set_ylabel('Training loss')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()
    


The base-trajectory plot above uses the same initialization for SGD and Muon within each network type. It does **not** by itself demonstrate convergence to the same solution or prove contraction; it just gives context for the perturbation analysis.
    


## 5. Face 2: independent-run diversity after fixed training

Face 2 trains many independent initializations for a fixed number of steps and then compares pairwise weight diversity, pairwise function diversity on `X_test`, and the spread of final losses.

This is now presented explicitly as a **fixed-budget training comparison**, not as a proof that either optimizer has converged.
    


In [ ]:
show_rows(results['summary_tables']['face2'], title='Face 2 summary metrics')
    


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, net_type in zip(axes, muon_exp.NETWORK_TYPES):
    sgd_runs = results['networks'][net_type]['face2']['sgd']['runs']
    muon_runs = results['networks'][net_type]['face2']['muon']['runs']

    sgd_losses = np.array([run['loss_trajectory'] for run in sgd_runs], dtype=float)
    muon_losses = np.array([run['loss_trajectory'] for run in muon_runs], dtype=float)

    for curve in sgd_losses:
        ax.plot(curve, color='#4477AA', alpha=0.15, linewidth=1)
    for curve in muon_losses:
        ax.plot(curve, color='#CC3311', alpha=0.15, linewidth=1)

    ax.plot(np.mean(sgd_losses, axis=0), color='#4477AA', linewidth=2.5, label='SGD mean')
    ax.plot(np.mean(muon_losses, axis=0), color='#CC3311', linewidth=2.5, label='Muon mean')
    ax.set_title(f'{net_type.upper()} net: per-run loss trajectories')
    ax.set_xlabel('Training step')
    ax.set_ylabel('Training loss')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()
    


The run-wise loss plot is useful context because the diversity metrics alone can over-compress the story. It shows whether the optimizer families are reaching broadly similar training-loss ranges under the current fixed training budget.
    


## 6. Heuristic checks and their limits

The script preserves the original T1-T6 checklist because it is part of the benchmark's current logic, but the notebook now labels it as a **heuristic summary** rather than a definitive statistical verdict.

Key limitation:
- The extra function-diversity separation statistic uses **pairwise distances that are not independent samples**, so it should not be treated as a formal significance test.
    


In [ ]:
show_rows(results['summary_tables']['heuristic_overview'], title='Heuristic overview by network')
show_rows(results['summary_tables']['heuristic_tests'], title='Heuristic test details')

for net_type in muon_exp.NETWORK_TYPES:
    sep = results['heuristics']['tests_by_net'][net_type]['heuristic_function_diversity_separation']
    display(Markdown(f'#### {net_type.upper()} heuristic separation note'))
    display(sep)
    


## 7. Calibrated conclusion
    


In [ ]:
heuristics = results['heuristics']
conclusion_md = f"""
### Support level: **{heuristics['support_label']}**

- Total heuristic passes: **{heuristics['total_pass']}/{heuristics['total_tests']}**
- Benchmark detail: {heuristics['detail']}

**Calibrated interpretation**

{heuristics['calibrated_conclusion']}

**What this notebook supports**
- In this toy setup, Muon can be associated with larger weight-space divergence than SGD while still yielding lower current function-space proxy divergence.
- The same toy pattern can also be summarized through lower function-diversity/weight-diversity ratios after fixed-budget training.

**What this notebook does not support**
- It does not prove that Muon is globally contractive in function space.
- It does not prove that the observed extra weight variation lies exactly in gauge directions.
- It does not establish a general paradox resolution or a mechanism proof.
"""
display(Markdown(conclusion_md))
    


## Raw-results note

The in-memory object `results` is the main notebook-facing API from the script. It includes:
- `config` and `seeds`
- `lr_selection`
- `networks[net_type]['face1']` with base trajectories, perturbation trajectories, and summary metrics
- `networks[net_type]['face2']` with per-run loss trajectories and diversity metrics
- `heuristics`
- `summary_tables`
- `artifacts`

That structure is intended to support future notebook refinements without re-duplicating the experiment core.
    
